In [1]:
import os
import pickle
import numpy as np
import pandas as pd

from scipy.signal import find_peaks, welch
from scipy.interpolate import interp1d


In [11]:
#This Cell contains code copied from notebook2 for creating windows and signal extraction functions
def create_windows(data, window_size, step_size):
    windows = []
    for start in range(0, len(data) - window_size + 1, step_size):
        windows.append(data[start:start + window_size])
    return windows

def extract_eda_features(window, fs=4):
    features = {}
    
    # Basic statistics
    features["mean"] = np.mean(window)
    features["std"] = np.std(window)
    features["min"] = np.min(window)
    features["max"] = np.max(window)
    
    # Linear slope (trend)
    x = np.arange(len(window))
    slope = np.polyfit(x, window, 1)[0]
    features["slope"] = slope
    
    # Peak detection (simple SCR estimation)
    peaks, _ = find_peaks(window, distance=fs)  
    features["num_peaks"] = len(peaks)
    
    return features

def compute_lfhf(ibi, fs_interp=4):
    # ibi in ms → convert to seconds
    t = np.cumsum(ibi) / 1000
    # remove first element to align
    t = t[:-1]
    ibi = ibi[1:]
    
    # Interpolation
    f = interp1d(t, ibi, kind='cubic', fill_value="extrapolate")
    t_interp = np.arange(t[0], t[-1], 1/fs_interp)
    ibi_interp = f(t_interp)
    
    # Compute power spectral density
    f_psd, p_psd = welch(ibi_interp, fs=fs_interp, nperseg=min(256, len(ibi_interp)))
    
    # LF band 0.04–0.15 Hz
    lf = np.trapezoid(p_psd[(f_psd >= 0.04) & (f_psd <= 0.15)], f_psd[(f_psd >= 0.04) & (f_psd <= 0.15)])
    # HF band 0.15–0.4 Hz
    hf = np.trapezoid(p_psd[(f_psd >= 0.15) & (f_psd <= 0.4)], f_psd[(f_psd >= 0.15) & (f_psd <= 0.4)])
    
    lf_hf = lf / hf if hf != 0 else 0
    return lf, hf, lf_hf

def extract_bvp_features(window, fs=64):
    features = {}
    
    # Detect peaks (heartbeat locations)
    peaks, _ = find_peaks(window, distance=fs*0.5)  # min 0.5 s between beats
    ibi = np.diff(peaks) / fs * 1000  # convert to ms
    
    # Heart rate
    hr = 60 / (ibi / 1000)  # bpm
    features['HR_mean'] = np.mean(hr)
    features['HR_std'] = np.std(hr)
    
    # HRV metrics time-domain
    features['RMSSD'] = np.sqrt(np.mean(np.diff(ibi)**2))
    features['pNN50'] = np.sum(np.abs(np.diff(ibi)) > 50) / len(ibi) * 100
    
    # LF/HF  simplified version
    # Here I can later do FFT for LF/HF
    # For now I will leave placeholders
    #features['LF'] = 0
   # features['HF'] = 0
    #features['LF_HF'] = 0

    
    # Frequency-domain HRV
    if len(ibi) > 4:  # need enough beats for PSD
        lf, hf, lf_hf = compute_lfhf(ibi)
        features['LF'] = lf
        features['HF'] = hf
        features['LF_HF'] = lf_hf
    else:
        features['LF'] = 0
        features['HF'] = 0
        features['LF_HF'] = 0
    return features


def extract_acc_features(window):
    features = {}

    # Split axes
    x = window[:, 0]
    y = window[:, 1]
    z = window[:, 2]

    # Per-axis statistics
    features['ACC_x_mean'] = np.mean(x)
    features['ACC_x_std'] = np.std(x)

    features['ACC_y_mean'] = np.mean(y)
    features['ACC_y_std'] = np.std(y)

    features['ACC_z_mean'] = np.mean(z)
    features['ACC_z_std'] = np.std(z)

    # Magnitude
    magnitude = np.sqrt(x**2 + y**2 + z**2)

    features['ACC_mag_mean'] = np.mean(magnitude)
    features['ACC_mag_std'] = np.std(magnitude)

    return features


def extract_temp_features(window):
    features = {}
    
    features['TEMP_mean'] = np.mean(window)
    features['TEMP_std'] = np.std(window)
    features['TEMP_min'] = np.min(window)
    features['TEMP_max'] = np.max(window)
    features['TEMP_slope'] = (window[-1] - window[0]) / len(window)
    
    return features

In [12]:
def process_subject(subject_path):
    
    # Load subject
    with open(subject_path, "rb") as file:
        data = pickle.load(file, encoding="latin1")
    
    wrist = data['signal']['wrist']
    
    bvp = wrist['BVP'].flatten()
    eda = wrist['EDA'].flatten()
    acc = wrist['ACC']
    temp = wrist['TEMP'].flatten()
    labels = data['label']
    
    # Create windows (reusing my existing window function)
    
    # BVP 
    bvp_windows = create_windows(bvp, 64*60, 64*60)
    
    # EDA
    eda_windows = create_windows(eda, 4*60, 4*60)
    
    # ACC
    acc_windows = create_windows(acc, 32*60, 32*60)
    
    # TEMP
    temp_windows = create_windows(temp, 4*60, 4*60)
    
    # Extract features
    bvp_df = pd.DataFrame([extract_bvp_features(win) for win in bvp_windows])
    eda_df = pd.DataFrame([extract_eda_features(win) for win in eda_windows])
    acc_df = pd.DataFrame([extract_acc_features(win) for win in acc_windows])
    temp_df = pd.DataFrame([extract_temp_features(win) for win in temp_windows])
    
    #  Combine features
    features_df = pd.concat([bvp_df, eda_df, acc_df, temp_df], axis=1)
    
    #  Window labels
    label_windows = create_windows(labels, 700*60, 700*60)  # WESAD label fs = 700Hz
    window_labels = [int(np.round(np.mean(win))) for win in label_windows]
    
    X_subject = features_df.values
    y_subject = np.array(window_labels)
    
    return X_subject, y_subject


In [17]:
wesad_path = "../data/WESAD"

subjects = [f for f in os.listdir(wesad_path) if f.startswith("S")]

all_X = []
all_y = []
subject_ids = []

for subject in subjects:
    
    subject_path = os.path.join(wesad_path, subject, f"{subject}.pkl")
    
    X_sub, y_sub = process_subject(subject_path)
    
    all_X.append(X_sub)
    all_y.append(y_sub)
    
    subject_ids.extend([subject] * len(y_sub))

X = np.vstack(all_X)
y = np.hstack(all_y)
subject_ids = np.array(subject_ids)

print("Final dataset shape:", X.shape)
print("Final label shape:", y.shape)
print("Unique labels:", np.unique(y))
print("Number of subjects:", len(subjects))


C:\Users\PC\AppData\Local\Temp\ipykernel_20064\3031582763.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(file, encoding="latin1")


Final dataset shape: (1442, 26)
Final label shape: (1442,)
Unique labels: [0 1 2 3 4 5 6 7]
Number of subjects: 15


In [20]:
#inspect distribution to see how many winows belong to which label
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Label {u}: {c}")


#clean dataset by filtering unwanted labels
valid_labels = [1, 2, 3, 4]

mask = np.isin(y, valid_labels)

X_clean = X[mask]
y_clean = y[mask]
subject_ids_clean = subject_ids[mask]

print("Clean dataset shape:", X_clean.shape)
print("Unique cleaned labels:", np.unique(y_clean))


Label 0: 583
Label 1: 340
Label 2: 201
Label 3: 108
Label 4: 195
Label 5: 9
Label 6: 4
Label 7: 2
Clean dataset shape: (844, 26)
Unique cleaned labels: [1 2 3 4]


In [ ]:
#Decided to first start with Binary classification instead of multiclass
# Map to binary
y_binary = np.where(y_clean == 2, 1, 0)

# Sanity check
unique, counts = np.unique(y_binary, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Label {u}: {c} windows")


Label 0: 643 windows
Label 1: 201 windows


In [ ]:
#preparing final dataset
X_final = X_clean
y_final = y_binary
subjects_final = subject_ids_clean

print("Final dataset shape:", X_final.shape)
print("Final labels shape:", y_final.shape)


Final dataset shape: (844, 26)
Final labels shape: (844,)
